# chesslm — full experiment run on a Colab GPU

Runs the complete tokenizer sweep (vocab 257 / 512 / 1024 / 2048): data prep, training,
legal-move evaluation, figures, then pushes `results/` and `figures/` back to GitHub.

**Before running:** `Runtime → Change runtime type → T4 GPU`, and add a Colab secret
(🔑 icon in the left sidebar) named `GITHUB_TOKEN` holding a GitHub fine-grained token with
*Contents: read and write* on this repo. Turn on *Notebook access* for the secret.

Data and checkpoints are kept on Google Drive, so a disconnect does not lose finished runs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = 'moon-silence/chess-transformer'
TOKEN = userdata.get('GITHUB_TOKEN')
!git clone -q https://{TOKEN}@github.com/{REPO}.git /content/chess-transformer 2>/dev/null || (cd /content/chess-transformer && git pull -q)
%cd /content/chess-transformer
!pip install -q -e .

In [ ]:
# Keep data/ and runs/ on Drive so they survive runtime resets.
import os
STORE = '/content/drive/MyDrive/chess-transformer'
for d in ['data', 'runs']:
    os.makedirs(f'{STORE}/{d}', exist_ok=True)
    if not os.path.islink(d):
        !rm -rf {d} && ln -s {STORE}/{d} {d}
!ls -la data runs

## 1. Data (≈10 min the first time, skipped afterwards)

In [ ]:
import os
if not os.path.exists('data/raw_pgn/Carlsen.pgn'):
    !bash scripts/download_data.sh
if not os.path.exists('data/v2048/meta.json'):
    !python -u scripts/prepare_data.py --vocab-sizes 257 512 1024 2048
!cat data/games/stats.json | head -8
!for v in 257 512 1024 2048; do python -c "import json; m=json.load(open('data/v$v/meta.json')); print($v, m['splits']['train'])"; done

## 2. Run the sweep (≈5–10 min per vocabulary on a T4)

In [ ]:
!python -u scripts/run_sweep.py --vocab-sizes 257 512 1024 2048

## 3. Figures

In [ ]:
!python scripts/make_figures.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figures/*.png')):
    display(Image(f))

## 4. Push results back to GitHub

In [ ]:
!git config user.name "Gregory Gestrin" && git config user.email "$(git log -1 --format=%ae)"
!git add results/ figures/ && git commit -q -m "Add Colab sweep results and figures" && git push -q origin main
!git log --oneline -3